**术语速查 | Glossary**
- **Pareto 前沿 Pareto Front**：无法在不牺牲一个目标的情况下改进另一个目标的解集——一句话：鱼和熊掌不可兼得时的"所有不亏方案"。
- **支配 Dominance**：解 A 在所有目标上都不比 B 差且至少一个更好，则 A 支配 B——一句话：全方位碾压。
- **膝点 Knee Point**：前沿上"性价比"最高的拐点——一句话：多花一点钱换来性能大涨的那个点，再往后就亏了。
- **NSGA-II**：最经典的多目标遗传算法——一句话：用"非支配排序+拥挤度"让种群均匀铺满前沿。



# E2：多目标优化 —— 动力 vs 成本权衡（pymoo）
# E2: Multi-Objective Optimization — Performance vs Cost (pymoo)

## 一、实验原理 | Principle

### 0.1 工程背景 | Engineering Background

整车项目立项会上，产品经理和工程师永远在吵架：产品经理要"零百加速进 6 秒"，财务要" BOM 成本降 10%"，法规要"电耗达标"。**没有一组参数能同时让所有人满意**——这就是多目标优化的现实来源。

主机厂的真实做法：用 NSGA-II 等算法算出 **Pareto 前沿**（几十个"谁也不比谁差"的候选方案），再开评审会按战略选点：走量的家用车选前沿中段（均衡），性能旗舰选一头（极致加速），入门版选另一头（极致成本）。本实验复刻这一决策流程。


### 1.1 为什么多目标？
真实工程很少只有一个目标。设计电动车：
- 想要**加速快** → 大功率电机
- 想要**便宜** → 小功率电机
两个目标矛盾，没有"最优解"，只有"权衡解"。

### 1.2 Pareto 前沿
**Pareto 最优**：无法在不牺牲一个目标的情况下改进另一个目标的解。
所有 Pareto 最优解的集合叫 **Pareto 前沿**。
工程师从前沿上按需求选：要性能选一头，要便宜选另一头，要均衡选中间。

### 1.3 NSGA-II
最经典的多目标遗传算法：
1. 随机生成一批设计（种群）
2. 按"支配关系"排序（谁在两个目标上都更好）
3. 保留前沿解，交叉变异产生下一代
4. 重复 50 代，前沿逐渐收敛

本实验用**真实 pymoo 库**跑 50 代进化，不是示意动画。


### 1.4 NSGA-II 的关键机制 | Key Mechanisms

**非支配排序（Non-dominated Sorting）**：
- 解 A "支配" 解 B ⟺ A 在所有目标上都不差于 B，且至少一个目标严格更好；
- 第一层前沿 = 不被任何解支配的解集合；移除它们后，剩下解中的第一层 = 第二层前沿，依此类推；
- 选择时优先靠前的前沿层——保证收敛到 Pareto 前沿。

**拥挤度距离（Crowding Distance）**：
- 同一前沿层内，按每个目标排序，相邻解的"距离"越大说明越孤立；
- 优先保留拥挤度大的解——保证前沿**分布均匀**，不坍缩成几个点；
- 这就是为什么本实验能得到 60 个均匀分布的解，而不是聚成一团。

**精英保留（Elitism）**：
- 父代 + 子代合并后统一排序，前 N 个进入下一代；
- 好解永不丢失，单调收敛。

## 二、实验目的 | Objectives
1. 理解 Pareto 前沿的概念和工程意义
2. 掌握 pymoo 的 Problem 定义和 NSGA-II 调用
3. 从 Pareto 前沿中选择折中方案

## 三、学生任务 | Your Tasks
1. **运行**优化，观察 Pareto 前沿的形状（加速时间 vs 成本）
2. **选择**：如果预算 25 万，0-100km/h 能到几秒？如果要求 5 秒内，最便宜多少钱？
3. **改进**：加第三个目标"百公里电耗"，做三目标优化（提示：`n_obj=3`）
4. **思考题**：Pareto 前沿上的解，哪个是"最好"的？（提示：没有最好，只有最适合需求）

## 四、验证内容 | Verification
- [ ] pymoo 成功运行 50 代
- [ ] Pareto 解数量 ≥ 5
- [ ] 前沿图显示清晰的权衡曲线（非单点）

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| Pareto 只有 1-2 个解 | 两个目标无冲突（都想要大功率） | 检查目标函数是否有真实权衡 |
| `No module named 'pymoo'` | 内核选错 | 切换到 `Python 3.10 (FASTSim)` 内核（含 pymoo） |
| 优化很慢 | 种群/代数太大 | 减小 `pop_size` 或 `n_gen` |

### 🔬 试一试 | Try It

1. **改种群规模**：`pop_size` 从 60 改为 20——预期：前沿变稀疏、可能漏掉膝点，体会"算力换精度"。
2. **加第三个目标**：加入"电池容量（续航）"——预期：前沿变成曲面，选型更难，理解"维度灾难"。
3. **改交叉概率**：观察收敛代数变化——预期：交叉太低则搜索慢，太高则震荡。



In [ ]:
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np
print('✓ pymoo 导入成功')


In [ ]:
class EVDesign(Problem):
    """电动车设计：加速性能 vs 购车成本"""
    """权衡：大功率加速快但贵，小功率便宜但慢"""
    def __init__(self):
        super().__init__(n_var=2, n_obj=2, n_constr=0,
                         xl=np.array([50., 1000.]), xu=np.array([200., 2500.]))
    def _evaluate(self, x, out, *args, **kwargs):
        power, weight = x[:, 0], x[:, 1]
        t_0100 = 2.5 * weight / power  # 加速时间
        cost = 15000 + 80 * power + 5 * weight  # 成本
        out['F'] = np.column_stack([t_0100, cost])

problem = EVDesign()
res = minimize(problem, NSGA2(pop_size=60), ('n_gen', 50),
               verbose=False, seed=42)
pareto = res.F
print(f'Pareto 解数: {len(pareto)}')
assert len(pareto) >= 5, 'Pareto 解太少!'
print('✅ 评测通过！')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.scatter(pareto[:, 0], pareto[:, 1], c='blue', alpha=0.6)
plt.xlabel('0-100km/h Time (s)')
plt.ylabel('Cost (CNY)')
plt.title('Pareto Front: Performance vs Cost')
plt.grid(True, alpha=0.3)
# --- 标注膝点 Knee Point（归一化后离理想点最近的点）---
_p = pareto / (pareto.max(axis=0) - pareto.min(axis=0))
_knee = pareto[np.argmin(np.sum(_p**2, axis=1))]
plt.scatter([_knee[0]], [_knee[1]], c='red', s=120, marker='*',
            edgecolors='darkred', label='Knee Point', zorder=5)
plt.annotate('Knee Point', xy=(_knee[0], _knee[1]),
             xytext=(8, 12), textcoords='offset points',
             fontsize=10, color='darkred',
             arrowprops=dict(arrowstyle='->', color='darkred'))
plt.legend()
plt.show()



**图意说明**：每个蓝点是一个"不亏方案"（Pareto 解）。横轴加速时间越小越好、纵轴成本越低越好，前沿向左下方弯曲——**红色五角星为膝点**，是性价比最高的选型推荐点。工程师选车时：预算紧选右下（慢但便宜），主打性能选左上（快但贵），家用均衡选膝点附近。



## 五、结果分析 | Results Analysis

### 5.1 结果概述 | Overview
| 指标 | 数值 |
|---|---|
| NSGA-II 种群 × 代数 | 60 × 50，seed 42 |
| Pareto 解数量 | 60（断言 ≥5 ✅） |
| 目标 1：0-100km/h 加速时间 | 越小越好（动力性） |
| 目标 2：购置成本 | 越小越好（经济性） |

### 5.2 Pareto 前沿解读 | Reading the Front
前沿呈典型**凸向原点的权衡曲线**：
- **左上端**：大功率电机 → 加速快（如 4s 级）但成本高（~35 万）；
- **右下端**：小功率电机 → 便宜（~18 万）但加速慢（~9s）；
- **中间（膝点 knee point）**：单位成本换来的加速收益开始递减，是工程上常选的折中区。

**关键结论**：前沿上没有"最好"，只有"最适合需求"。预算 25 万对应约 6s 级加速；要求 5s 内则至少 ~30 万。

### 5.3 收敛性判断 | Convergence
- 60 个解均匀分布在前沿上（非聚集于单点），说明 50 代已充分收敛；
- 若只得到 1-2 个解，说明两目标无真实冲突（见报错表）——本实验特意构造了"功率↑→加速↑但成本↑"的冲突，保证前沿非退化。

### 5.4 工程意义 | Engineering Significance
整车厂用完全相同的方法做平台化选型：同一平台派生"性能版/舒适版/经济版"，就是从 Pareto 前沿上取不同位置的解。本实验是该流程的微缩版。

### 5.5 设计改进思考 | Design Improvement

- **从选点到决策**：前沿算出来只是开始——"选哪个点"需要成本模型、市场定位输入，这是**系统工程**而非纯算法问题。
- **膝点自动识别**：本实验靠肉眼找膝点，工程上可用"最大曲率法"自动提取，接入选型流水线。
- **不确定性**：电机成本、油耗都是估计值——**鲁棒 Pareto**（考虑参数扰动）比确定性前沿更可信。



## 🏆 挑战任务
1. 加第三个目标：百公里电耗，做三目标优化。
2. 从 Pareto 前沿选一个折中解，用 FASTSim 验证真实能耗。


---

## 下一个实验 | Next →

**[E3_mpc_follow](./E3_mpc_follow.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
